## Week 2 Day 1

いよいよ、OpenAI Agents SDKを初めて見ていきます

これがどれほど軽量なものか、きっと驚くはずです。

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">OpenAI Agents SDK のドキュメント</h2>
            <span style="color:#00bfff;">OpenAI Agents SDKのドキュメントは本当に分かりやすくシンプルです。<a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> 一度見てみる価値があります。
            </span>
        </td>
    </tr>
</table>

# このラボの3つのパート

## パート1: シンプルな「Agent」と「Agent Loop」

基本的にはLLMの呼び出しです。トレーシングとストリーミングも加えていきます。

## パート2: ツールの追加

おなじみのものですが、驚くほど簡単です

## パート3: メモリの追加

異なるAgentの呼び出し同士が、互いを認識できるようにします

In [ ]:
# インポート

import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
load_dotenv(override=True)


## 補足

このフレームワークの、Python公式インデックスpypi.org上での正式名称は`openai-agents`です。

そのため、今後自分のプロジェクトでは、次のようにします。

`pip install openai-agents`  
または  
`uv add openai-agents`

そのうえで

`from agents import Agent, Runner, trace`

`pip install agents`を実行すると、まったく別のもの（古い強化学習ライブラリ）がインストールされてしまうので注意してください。


In [ ]:

# name、instructions、modelを指定してエージェントを作成する

agent = Agent(name="Jokester", instructions="You are a joke teller", model="gpt-5.4-mini")

In [ ]:
# Runner.run(agent, prompt)でジョークを実行する

result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")


In [ ]:
# こちらが最終出力です

print(result.final_output)

In [ ]:
# こちらがLLM呼び出しの詳細です

result.to_input_list()

## traceによる可観測性（Observability）の追加

In [ ]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)

## さあ、traceを見てみましょう

https://platform.openai.com/traces

In [ ]:
# ストリーミング

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

## パート2: ツールの追加

In [ ]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

In [ ]:
# これを覚えていますか？

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
push("HEY!!")

In [ ]:
push

In [ ]:
# 今度はこうします:

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [ ]:
push_tool

In [ ]:
push_tool.description

In [ ]:

notifier = Agent(name="Notifier", model="gpt-5.4-mini", instructions="You notify the user upon request", tools=[push_tool])

In [ ]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)


## さあ、traceを見てみましょう

https://platform.openai.com/traces

## パート3: セッション（メモリ）

Runner.run()のアプリケーションレベルの1ターンの中では、会話履歴が保持されます。

しかし、Runner.run()を呼び出すたびに、それは新しい始まりになります。

それを確認してみましょう。

In [ ]:
agent = Agent(name="Assistant", model="gpt-5.4-mini")

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

In [ ]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

## メモリのやり方その1 - dictのリストを手動で渡す

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

In [ ]:
response.to_input_list()

In [ ]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

In [ ]:
response = await Runner.run(agent, next_input)
print(response.final_output)

## 別のやり方 - OpenAI Agents SDKに組み込まれているSQLiteセッションを使う

In [ ]:
# これはインメモリで作成されます
# ディスク上に保存するメモリにするには、SQLiteSession("12345", "memory.db")を使います

session = SQLiteSession("12346")

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Ed.", session=session)
print(response.final_output)

In [ ]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

# すごい！

ラボ1でここまでできたなんて信じられますか？！

Agents、Runner（Agent Loop）、traces（可観測性）、ストリーミング、Function Tools、メモリ！

ぜひドキュメントもチェックしてください。  
https://openai.github.io/openai-agents-python/

さらに良い知らせがあります。軽量なAgentフレームワークの多くは非常によく似ているので、これでほぼすべてを理解したことになります。


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">演習</h2>
            <span style="color:#ff7800;">OpenAI Agents SDKを使って、Week 1のプロジェクトのいずれか（デジタルツインやChecklistループなど）を作ってみましょう。どれほど簡単か、きっと驚くはずです。
            </span>
        </td>
    </tr>
</table>